# Notebook 1: Constructing the Curated Thesis Corpus

The goal of this notebook is to move away from the massive, noisy raw database and build a clean, defensible corpus specifically tailored for this thesis. Rather than carrying around unverified data, we systematically filter the raw TMDB extract down to fictional, feature-length, released films within a stable historical window.

Most importantly, we ensure every film retained has usable plot text for the NLP model and sufficient audience engagement data.

Ultimately, this pipeline generates four focused subsets:
*   `df_thesis_films.csv` - The baseline cleaned corpus for general EDA.
*   `df_semantic_analysis.csv` - A stricter subset with high-quality text, reserved exclusively for the SBERT embeddings and network construction.
*   `df_revenue_analysis.csv` - Films with verified commercial revenue.
*   `df_roi_analysis.csv` - The tightest subset containing both budget and revenue figures for financial modeling.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Directory Setup and Global Parameters
Defining the file paths and the core boundaries for the thesis sample (e.g., release years, minimum runtimes, and vote thresholds). If the scope of the thesis needs to be adjusted later, these are the only parameters that need to change.

In [ ]:
BASE_DIR = "/content/drive/MyDrive/thesis_movie_project"

RAW_DIR = f"{BASE_DIR}/raw"
PROCESSED_CORE_DIR = f"{BASE_DIR}/processed/core"
PROCESSED_FEATURES_DIR = f"{BASE_DIR}/processed/features"
OUTPUT_TABLES_DIR = f"{BASE_DIR}/outputs/tables"

RAW_FILENAME = "TMDB_all_movies.csv"
RAW_PATH = f"{RAW_DIR}/{RAW_FILENAME}"

LOCAL_RAW_DIR = "/content/thesis_movie_project/raw"
LOCAL_RAW_PATH = f"{LOCAL_RAW_DIR}/{RAW_FILENAME}"

os.makedirs(PROCESSED_CORE_DIR, exist_ok=True)
os.makedirs(PROCESSED_FEATURES_DIR, exist_ok=True)
os.makedirs(OUTPUT_TABLES_DIR, exist_ok=True)
os.makedirs(LOCAL_RAW_DIR, exist_ok=True)

# Thesis sample definition
YEAR_MIN = 1990
YEAR_MAX = 2024

MIN_RUNTIME = 70
MAX_RUNTIME = 240

MIN_OVERVIEW_WORDS_THESIS = 25
MIN_OVERVIEW_WORDS_SEMANTIC = 35

MIN_VOTE_COUNT_THESIS = 1       # low threshold: keeps broad but not dead rows
MIN_VOTE_COUNT_SEMANTIC = 25    # stricter network corpus
MIN_VOTE_COUNT_DISPLAY = 50     # only for thesis tables, not graph construction

EXCLUDE_TV_MOVIE = True
EXCLUDE_DOCUMENTARY = True

print("Raw path:", RAW_PATH)
print("Main thesis window:", YEAR_MIN, "-", YEAR_MAX)


Raw path: /content/drive/MyDrive/thesis_movie_project/raw/TMDB_all_movies.csv
Main thesis window: 1990 - 2024


In [ ]:
if not os.path.exists(RAW_PATH):
    raise FileNotFoundError(f"Raw file not found: {RAW_PATH}")

needs_copy = True
if os.path.exists(LOCAL_RAW_PATH):
    try:
        needs_copy = os.path.getsize(RAW_PATH) != os.path.getsize(LOCAL_RAW_PATH)
    except OSError:
        needs_copy = True

if needs_copy:
    print("Copying raw file from Drive to local Colab storage...")
    shutil.copy2(RAW_PATH, LOCAL_RAW_PATH)
else:
    print("Local raw copy already exists and file size matches.")

print("Local raw file:", LOCAL_RAW_PATH)


Local raw copy already exists and file size matches.
Local raw file: /content/thesis_movie_project/raw/TMDB_all_movies.csv


## 2. Text Cleaning and Parsing Utilities
A few helper functions to handle messy text columns, normalize movie titles for deduplication, and safely parse stringified lists (like genres and cast arrays) from the raw CSV.

In [ ]:
def clean_text(series):
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def to_num(series):
    return pd.to_numeric(series, errors="coerce")

def pick_col(columns, candidates, required=False):
    for c in candidates:
        if c in columns:
            return c
    if required:
        raise KeyError(f"None of these required columns were found: {candidates}")
    return None

def parse_comma_or_listlike(x, top_n=None):
    """
    Handles:
    - "Action, Drama"
    - "['Action', 'Drama']"
    - empty/missing values
    """
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null", "[]"}:
        return []

    quoted = re.findall(r"'([^']+)'|\"([^\"]+)\"", s)
    if quoted:
        vals = []
        for a, b in quoted:
            vals.append((a or b).strip())
    else:
        vals = [p.strip(" []'\"") for p in s.split(",")]

    vals = [v for v in vals if v and v.lower() not in {"nan", "none", "null"}]
    return vals[:top_n] if top_n else vals

def clean_list_column(series, top_n=None, separator=", "):
    return series.apply(lambda x: separator.join(parse_comma_or_listlike(x, top_n=top_n)))

def normalize_title(series):
    s = clean_text(series).str.lower()
    s = s.str.replace(r"[^a-z0-9\s]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

def show_funnel(funnel_rows):
    df = pd.DataFrame(funnel_rows)
    df["rows_removed"] = df["rows_before"] - df["rows_after"]
    df["pct_kept_from_previous"] = (df["rows_after"] / df["rows_before"] * 100).round(2)
    return df


## 3. Importing the Raw Data
To keep memory usage manageable and avoid a bloated master file, we only pull in the specific raw columns required for the network construction and financial analysis.

In [ ]:
raw_head = pd.read_csv(LOCAL_RAW_PATH, nrows=5, low_memory=False)
print("Available raw columns:")
print(list(raw_head.columns))

cols = list(raw_head.columns)

col_id = pick_col(cols, ["id"], required=True)
col_title = pick_col(cols, ["title"], required=True)
col_imdb_id = pick_col(cols, ["imdb_id"])
col_overview = pick_col(cols, ["overview"], required=True)
col_release_date = pick_col(cols, ["release_date"], required=True)
col_status = pick_col(cols, ["status"])
col_runtime = pick_col(cols, ["runtime"])
col_genres = pick_col(cols, ["genres", "genres_list"])
col_cast = pick_col(cols, ["cast", "Cast_list"])
col_director = pick_col(cols, ["director", "Director", "director_name"])
col_language = pick_col(cols, ["original_language"])
col_budget = pick_col(cols, ["budget"])
col_revenue = pick_col(cols, ["revenue"])
col_popularity = pick_col(cols, ["popularity"])
col_vote_average = pick_col(cols, ["vote_average", "imdb_rating"])
col_vote_count = pick_col(cols, ["vote_count", "imdb_votes"])

usecols = [c for c in [
    col_id, col_title, col_imdb_id, col_overview, col_release_date, col_status,
    col_runtime, col_genres, col_cast, col_director, col_language, col_budget,
    col_revenue, col_popularity, col_vote_average, col_vote_count
] if c is not None]

usecols = sorted(set(usecols))
print("\nSelected raw columns:")
print(usecols)

raw = pd.read_csv(LOCAL_RAW_PATH, usecols=usecols, low_memory=False)
print("\nRaw loaded shape:", raw.shape)
display(raw.head(3))


Available raw columns:
['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'budget', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'cast', 'director', 'director_of_photography', 'writers', 'producers', 'music_composer', 'imdb_rating', 'imdb_votes', 'poster_path']

Selected raw columns:
['budget', 'cast', 'director', 'genres', 'id', 'imdb_id', 'original_language', 'overview', 'popularity', 'release_date', 'revenue', 'runtime', 'status', 'title', 'vote_average', 'vote_count']

Raw loaded shape: (1192309, 16)


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,original_language,overview,popularity,genres,cast,director
0,2,Ariel,7.106,371.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,fi,A Finnish man goes to the city to find a job a...,1.6384,"Comedy, Drama, Romance, Crime","Kari Helaseppä, Jaakko Talaskivi, Mikko Remes,...",Aki Kaurismäki
1,3,Shadows in Paradise,7.300,435.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,fi,"Nikander, a rubbish collector and would-be ent...",2.7136,"Comedy, Drama, Romance","Ari Korhonen, Mari Rantasila, Erkki Rissanen, ...",Aki Kaurismäki
2,5,Four Rooms,5.900,2821.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,en,It's Ted the Bellhop's first night on the job....,4.0661,Comedy,"David Proval, Tamlyn Tomita, Paul Skemp, Lana ...","Quentin Tarantino, Robert Rodriguez, Alexandre..."


## 4. Standardizing the Working Dataframe
Here we enforce correct data types, handle null values, and engineer the `text_semantic_plot` column. This column deliberately stitches together the title, genre, and plot overview while excluding actor/director names—this ensures the NLP model clusters films by narrative tropes rather than casting similarities.

In [ ]:
df = pd.DataFrame(index=raw.index)

df["id"] = to_num(raw[col_id]).astype("Int64")
df["imdb_id"] = clean_text(raw[col_imdb_id]) if col_imdb_id else ""
df["title"] = clean_text(raw[col_title])
df["overview"] = clean_text(raw[col_overview])
df["release_date"] = pd.to_datetime(raw[col_release_date], errors="coerce")
df["release_year"] = df["release_date"].dt.year.astype("Int64")

df["status"] = clean_text(raw[col_status]) if col_status else ""
df["runtime"] = to_num(raw[col_runtime]) if col_runtime else np.nan

df["genres_clean"] = clean_list_column(raw[col_genres]) if col_genres else ""
df["cast_clean"] = clean_list_column(raw[col_cast], top_n=5) if col_cast else ""
df["director_name"] = clean_text(raw[col_director]) if col_director else ""
df["original_language"] = clean_text(raw[col_language]) if col_language else ""

df["budget"] = to_num(raw[col_budget]) if col_budget else np.nan
df["revenue"] = to_num(raw[col_revenue]) if col_revenue else np.nan
df["popularity"] = to_num(raw[col_popularity]) if col_popularity else np.nan
df["vote_average"] = to_num(raw[col_vote_average]) if col_vote_average else np.nan
df["vote_count"] = to_num(raw[col_vote_count]) if col_vote_count else np.nan

# Convert placeholder strings to empty strings
for c in ["imdb_id", "title", "overview", "status", "genres_clean", "cast_clean", "director_name", "original_language"]:
    df[c] = df[c].replace({
        "nan": "", "NaN": "", "None": "", "NONE": "", "null": "", "NULL": "", "[]": "", "Unknown": "", "N/A": ""
    })

df["overview_word_count"] = df["overview"].str.split().str.len().fillna(0).astype(int)

df["budget_clean"] = df["budget"].where(df["budget"] > 0)
df["revenue_clean"] = df["revenue"].where(df["revenue"] > 0)
df["roi_clean"] = np.where(
    df["budget_clean"].notna() & df["revenue_clean"].notna(),
    df["revenue_clean"] / df["budget_clean"],
    np.nan
)
df["log_revenue"] = np.where(df["revenue_clean"].notna(), np.log1p(df["revenue_clean"]), np.nan)

df["primary_genre"] = df["genres_clean"].str.split(",").str[0].str.strip()
df["is_tv_movie"] = df["genres_clean"].str.contains(r"\bTV Movie\b", regex=True, na=False)

df["text_overview_only"] = df["overview"]
df["text_semantic_plot"] = (
    df["title"] + ". Genres: " + df["genres_clean"] + ". Plot: " + df["overview"]
).str.replace(r"\s+", " ", regex=True).str.strip()

df["source_name"] = RAW_FILENAME

print("Compact standardized shape:", df.shape)
display(df.head(3))


Compact standardized shape: (1192309, 27)


,id,imdb_id,title,overview,release_date,release_year,status,runtime,genres_clean,cast_clean,director_name,original_language,budget,revenue,popularity,vote_average,vote_count,overview_word_count,budget_clean,revenue_clean,roi_clean,log_revenue,primary_genre,is_tv_movie,text_overview_only,text_semantic_plot,source_name
0,2,tt0094675,Ariel,A Finnish man goes to the city to find a job a...,1988-10-21,1988,Released,73.0,"Comedy, Drama, Romance, Crime","Kari Helaseppä, Jaakko Talaskivi, Mikko Remes,...",Aki Kaurismäki,fi,0.0,0.0,1.6384,7.106,371.0,24,NaN,NaN,NaN,NaN,Comedy,False,A Finnish man goes to the city to find a job a...,"Ariel. Genres: Comedy, Drama, Romance, Crime. ...",TMDB_all_movies.csv
1,3,tt0092149,Shadows in Paradise,"Nikander, a rubbish collector and would-be ent...",1986-10-17,1986,Released,74.0,"Comedy, Drama, Romance","Ari Korhonen, Mari Rantasila, Erkki Rissanen, ...",Aki Kaurismäki,fi,0.0,0.0,2.7136,7.300,435.0,38,NaN,NaN,NaN,NaN,Comedy,False,"Nikander, a rubbish collector and would-be ent...","Shadows in Paradise. Genres: Comedy, Drama, Ro...",TMDB_all_movies.csv
2,5,tt0113101,Four Rooms,It's Ted the Bellhop's first night on the job....,1995-12-09,1995,Released,98.0,Comedy,"David Proval, Tamlyn Tomita, Paul Skemp, Lana ...","Quentin Tarantino, Robert Rodriguez, Alexandre...",en,4000000.0,4257354.0,4.0661,5.900,2821.0,38,4000000.0,4257354.0,1.064339,15.264159,Comedy,False,It's Ted the Bellhop's first night on the job....,Four Rooms. Genres: Comedy. Plot: It's Ted the...,TMDB_all_movies.csv


## 5. The Filtering Funnel
This is the core of the data construction. We systematically drop unreleased projects, short films, TV movies, documentaries, and entries with less than 25 words in their overview. Every step is tracked in a funnel log so we have a transparent, academic record of exactly why records were excluded.

In [ ]:
funnel = []
work = df.copy()

def apply_filter(work, step, condition):
    before = len(work)
    out = work.loc[condition].copy()
    funnel.append({"step": step, "rows_before": before, "rows_after": len(out)})
    return out

# Initial count
funnel.append({"step": "raw_loaded_selected_columns", "rows_before": len(work), "rows_after": len(work)})

work = apply_filter(
    work,
    "released_status_only",
    (work["status"].eq("")) | (work["status"].str.lower().eq("released"))
)

today = pd.Timestamp.today().normalize()
work = apply_filter(
    work,
    "not_future_release",
    work["release_date"].notna() & (work["release_date"] <= today)
)

work = apply_filter(
    work,
    f"stable_year_window_{YEAR_MIN}_{YEAR_MAX}",
    work["release_year"].between(YEAR_MIN, YEAR_MAX, inclusive="both")
)

work = apply_filter(
    work,
    "usable_title",
    work["title"].str.len() > 0
)

work = apply_filter(
    work,
    f"feature_runtime_{MIN_RUNTIME}_{MAX_RUNTIME}_minutes",
    work["runtime"].between(MIN_RUNTIME, MAX_RUNTIME, inclusive="both")
)

work = apply_filter(
    work,
    f"overview_at_least_{MIN_OVERVIEW_WORDS_THESIS}_words",
    work["overview_word_count"] >= MIN_OVERVIEW_WORDS_THESIS
)

work = apply_filter(
    work,
    "has_genre",
    work["genres_clean"].str.len() > 0
)

if EXCLUDE_TV_MOVIE:
    work = apply_filter(
        work,
        "exclude_tv_movie_genre",
        ~work["is_tv_movie"]
    )


if EXCLUDE_DOCUMENTARY:
    work = apply_filter(
        work,
        "exclude_documentary_primary_genre",
        work["primary_genre"].ne("Documentary")
    )

work = apply_filter(
    work,
    f"minimum_vote_count_{MIN_VOTE_COUNT_THESIS}",
    work["vote_count"].fillna(0) >= MIN_VOTE_COUNT_THESIS
)

funnel_df = show_funnel(funnel)
display(funnel_df)
print("Thesis corpus before dedupe:", work.shape)


,step,rows_before,rows_after,rows_removed,pct_kept_from_previous
0,raw_loaded_selected_columns,1192309,1192309,0,100.00
1,released_status_only,1192309,1170497,21812,98.17
2,not_future_release,1170497,1063713,106784,90.88
3,stable_year_window_1990_2024,1063713,727301,336412,68.37
4,usable_title,727301,727281,20,100.00
5,feature_runtime_70_240_minutes,727281,258798,468483,35.58
6,overview_at_least_25_words,258798,200002,58796,77.28
7,has_genre,200002,180239,19763,90.12
8,exclude_tv_movie_genre,180239,170025,10214,94.33
9,exclude_documentary_primary_genre,170025,134792,35233,79.28


Thesis corpus before dedupe: (98038, 27)


## 6. Quality-Weighted Deduplication
Handling duplicates without blindly dropping rows. We check for exact TMDB ID matches, then IMDb IDs, and finally a combination of title and release year. When duplicates are found, we preserve the row that contains the strongest metadata (the highest vote counts, popularity, and revenue).

In [ ]:
dedup_log = []

work = work.copy()
work["title_key"] = normalize_title(work["title"])
work["title_year_key"] = np.where(
    (work["title_key"].str.len() > 0) & (work["release_year"].notna()),
    work["title_key"] + "||" + work["release_year"].astype(str),
    np.nan
)

work["dedupe_quality"] = (
    work["vote_count"].fillna(0).clip(upper=100000) * 0.001 +
    work["popularity"].fillna(0).clip(upper=10000) * 0.01 +
    work["overview_word_count"].fillna(0).clip(upper=200) * 0.05 +
    work["revenue_clean"].notna().astype(int) * 10 +
    work["budget_clean"].notna().astype(int) * 5 +
    work["imdb_id"].str.len().gt(0).astype(int) * 3
)

def log_step(step, before, after):
    dedup_log.append({
        "step": step,
        "rows_before": before,
        "rows_after": after,
        "rows_removed": before - after
    })

# 1. ID dedupe
before = len(work)
with_id = work[work["id"].notna()].sort_values(["id", "dedupe_quality"], ascending=[True, False])
without_id = work[work["id"].isna()]
with_id = with_id.drop_duplicates("id", keep="first")
work = pd.concat([with_id, without_id], ignore_index=True)
log_step("dedupe_exact_tmdb_id", before, len(work))

# 2. IMDb ID dedupe
before = len(work)
with_imdb = work[work["imdb_id"].str.len() > 0].sort_values(["imdb_id", "dedupe_quality"], ascending=[True, False])
without_imdb = work[work["imdb_id"].str.len() == 0]
with_imdb = with_imdb.drop_duplicates("imdb_id", keep="first")
work = pd.concat([with_imdb, without_imdb], ignore_index=True)
log_step("dedupe_exact_imdb_id", before, len(work))

# 3. title-year only when IMDb id missing
before = len(work)
ty = work[
    (work["imdb_id"].str.len() == 0) &
    (work["title_year_key"].notna())
].sort_values(["title_year_key", "dedupe_quality"], ascending=[True, False])

rest = work.drop(ty.index)
ty = ty.drop_duplicates("title_year_key", keep="first")
work = pd.concat([rest, ty], ignore_index=True)
log_step("dedupe_title_year_missing_imdb_only", before, len(work))

dedup_log_df = pd.DataFrame(dedup_log)
display(dedup_log_df)

# Sanity checks
assert work.loc[work["id"].notna(), "id"].duplicated().sum() == 0
assert work.loc[work["imdb_id"].str.len() > 0, "imdb_id"].duplicated().sum() == 0

print("Thesis corpus after dedupe:", work.shape)


,step,rows_before,rows_after,rows_removed
0,dedupe_exact_tmdb_id,98038,98038,0
1,dedupe_exact_imdb_id,98038,98038,0
2,dedupe_title_year_missing_imdb_only,98038,98037,1


Thesis corpus after dedupe: (98037, 30)


## 7. The Final Thesis Corpus
Trimming down the working dataframe into the final, streamlined master table. This serves as the foundation for the rest of the project.

In [ ]:
# Final compact thesis corpus columns
# Rename TMDB/source id clearly before exporting
work = work.rename(columns={"id": "tmdb_id"})

FINAL_THESIS_COLUMNS = [
    "tmdb_id",
    "imdb_id",
    "title",
    "original_language",
    "release_date",
    "release_year",
    "primary_genre",
    "genres_clean",
    "overview",
    "overview_word_count",
    "text_overview_only",
    "text_semantic_plot",
    "runtime",
    "cast_clean",
    "director_name",
    "budget_clean",
    "revenue_clean",
    "roi_clean",
    "log_revenue",
    "popularity",
    "vote_average",
    "vote_count",
    "status",
    "is_tv_movie"
]

df_thesis_films = work[FINAL_THESIS_COLUMNS].copy()

print("Final df_thesis_films shape:", df_thesis_films.shape)
print("Columns:", len(df_thesis_films.columns))
display(df_thesis_films.head(5))

Final df_thesis_films shape: (98037, 24)
Columns: 24


,tmdb_id,imdb_id,title,original_language,release_date,release_year,primary_genre,genres_clean,overview,overview_word_count,text_overview_only,text_semantic_plot,runtime,cast_clean,director_name,budget_clean,revenue_clean,roi_clean,log_revenue,popularity,vote_average,vote_count,status,is_tv_movie
0,286375,tt0015724,Dama de Noche,es,1993-05-18,1993,Drama,"Drama, Mystery, Romance, Thriller","Bruno, a novelist with no luck goes to call fo...",43,"Bruno, a novelist with no luck goes to call fo...","Dama de Noche. Genres: Drama, Mystery, Romance...",96.0,"Cecilia Toussaint, Bruno Bichir, Miguel Córceg...",Eva López Sánchez,NaN,NaN,NaN,NaN,0.0602,6.9,7.0,Released,False
1,11232,tt0035423,Kate & Leopold,en,2001-12-25,2001,Romance,"Romance, Comedy, Fantasy",When her scientist ex-boyfriend discovers a po...,64,When her scientist ex-boyfriend discovers a po...,"Kate & Leopold. Genres: Romance, Comedy, Fanta...",118.0,"Del Pentecost, Andrea Barnes, Philip Bosco, Br...",James Mangold,48000000.0,76019048.0,1.58373,18.146495,3.9819,6.3,1468.0,Released,False
2,565997,tt0055050,Das Kleid,de,1991-02-09,1991,Fantasy,"Fantasy, Comedy",Poor weavers Hans und Kumpan try to enter a to...,77,Poor weavers Hans und Kumpan try to enter a to...,"Das Kleid. Genres: Fantasy, Comedy. Plot: Poor...",88.0,"Nico Turoff, Axel Triebel, Horst Drinda, Wolf ...","Konrad Petzold, Egon Günther",NaN,NaN,NaN,NaN,1.9901,8.0,1.0,Released,False
3,257719,tt0064222,"The Seventh Day, the Eighth Night",cs,1990-06-01,1990,Drama,"Drama, Mystery",An allegory set in an archetypal Czech village...,82,An allegory set in an archetypal Czech village...,"The Seventh Day, the Eighth Night. Genres: Dra...",103.0,"Jaroslav Wagner-Klenka, Jiří Hálek, Josef Bek,...",Evald Schorm,NaN,NaN,NaN,NaN,0.2731,4.0,10.0,Released,False
4,611775,tt0064322,The Woman with the Knife,fr,2010-02-05,2010,Drama,"Drama, Thriller","A young, unnamed Ivorian intellectual returns ...",50,"A young, unnamed Ivorian intellectual returns ...","The Woman with the Knife. Genres: Drama, Thril...",77.0,"Tim Sory, Danielle Alloh, Bertin Kouakou, Timi...",Timité Bassori,NaN,NaN,NaN,NaN,0.3029,5.0,7.0,Released,False


## 8. Generating the Focused Datasets
Splitting the master corpus into the specific subsets required for the upcoming notebooks. The semantic dataset applies stricter word count and vote thresholds, while the financial datasets strictly require non-null monetary figures.

In [ ]:
df_semantic_analysis = df_thesis_films[
    (df_thesis_films["overview_word_count"] >= MIN_OVERVIEW_WORDS_SEMANTIC) &
    (df_thesis_films["vote_count"].fillna(0) >= MIN_VOTE_COUNT_SEMANTIC)
].copy()

SEMANTIC_COLUMNS = [
    "tmdb_id",
    "imdb_id",
    "title",
    "release_year",
    "primary_genre",
    "genres_clean",
    "overview",
    "overview_word_count",
    "text_semantic_plot",
    "runtime",
    "popularity",
    "vote_average",
    "vote_count",
    "revenue_clean",
    "log_revenue",
    "cast_clean",
    "director_name"
]

df_semantic_analysis = df_semantic_analysis[SEMANTIC_COLUMNS].copy()

df_display_candidates = df_semantic_analysis[
    df_semantic_analysis["vote_count"].fillna(0) >= MIN_VOTE_COUNT_DISPLAY
].copy()

df_revenue_analysis = df_thesis_films[
    df_thesis_films["revenue_clean"].notna()
].copy()

df_roi_analysis = df_thesis_films[
    df_thesis_films["revenue_clean"].notna() &
    df_thesis_films["budget_clean"].notna() &
    df_thesis_films["roi_clean"].notna() &
    (df_thesis_films["roi_clean"] > 0) &
    (df_thesis_films["roi_clean"] < 100)
].copy()

summary = pd.DataFrame([
    {"dataset": "df_thesis_films", "rows": len(df_thesis_films), "cols": df_thesis_films.shape[1]},
    {"dataset": "df_semantic_analysis", "rows": len(df_semantic_analysis), "cols": df_semantic_analysis.shape[1]},
    {"dataset": "df_display_candidates", "rows": len(df_display_candidates), "cols": df_display_candidates.shape[1]},
    {"dataset": "df_revenue_analysis", "rows": len(df_revenue_analysis), "cols": df_revenue_analysis.shape[1]},
    {"dataset": "df_roi_analysis", "rows": len(df_roi_analysis), "cols": df_roi_analysis.shape[1]},
])

display(summary)


,dataset,rows,cols
0,df_thesis_films,98037,24
1,df_semantic_analysis,20091,17
2,df_display_candidates,13965,17
3,df_revenue_analysis,13186,24
4,df_roi_analysis,7933,24


## 9. Final Corpus Audit
Generating summary statistics for the final datasets to ensure the timeline, runtime distributions, and missing values align with our expectations before moving into the network phase.

In [ ]:
year_counts = (
    df_thesis_films["release_year"]
    .value_counts()
    .sort_index()
    .rename_axis("release_year")
    .reset_index(name="movie_count")
)

genre_counts = (
    df_thesis_films["primary_genre"]
    .replace("", np.nan)
    .dropna()
    .value_counts()
    .head(30)
    .rename_axis("primary_genre")
    .reset_index(name="movie_count")
)

audit = pd.DataFrame([
    {"metric": "raw_rows_loaded", "value": len(df)},
    {"metric": "final_thesis_rows", "value": len(df_thesis_films)},
    {"metric": "final_thesis_cols", "value": df_thesis_films.shape[1]},
    {"metric": "year_min", "value": int(df_thesis_films["release_year"].min())},
    {"metric": "year_max", "value": int(df_thesis_films["release_year"].max())},
    {"metric": "median_runtime", "value": float(df_thesis_films["runtime"].median())},
    {"metric": "median_overview_words", "value": float(df_thesis_films["overview_word_count"].median())},
    {"metric": "median_vote_count", "value": float(df_thesis_films["vote_count"].median())},
    {"metric": "pct_revenue_available", "value": round(df_thesis_films["revenue_clean"].notna().mean() * 100, 2)},
    {"metric": "pct_budget_available", "value": round(df_thesis_films["budget_clean"].notna().mean() * 100, 2)},
    {"metric": "semantic_rows", "value": len(df_semantic_analysis)},
    {"metric": "display_candidate_rows_vote_ge_50", "value": len(df_display_candidates)},
])

display(audit)
display(year_counts.tail(15))
display(genre_counts.head(15))


,metric,value
0,raw_rows_loaded,1192309.00
1,final_thesis_rows,98037.00
2,final_thesis_cols,24.00
3,year_min,1990.00
4,year_max,2024.00
5,median_runtime,96.00
6,median_overview_words,52.00
7,median_vote_count,7.00
8,pct_revenue_available,13.45
9,pct_budget_available,15.28


,release_year,movie_count
20,2010,3127
21,2011,3349
22,2012,3475
23,2013,3799
24,2014,4000
25,2015,4097
26,2016,4251
27,2017,4483
28,2018,4591
29,2019,4641


,primary_genre,movie_count
0,Drama,31837
1,Comedy,19805
2,Action,8972
3,Horror,8490
4,Thriller,5290
5,Romance,4409
6,Music,4219
7,Crime,3249
8,Animation,2476
9,Family,2040


## 10. Exporting Core Files
Saving the curated datasets, the filtering funnel log, and the audit summaries to the processed directory.

In [ ]:
THESIS_FILMS_OUT = f"{PROCESSED_CORE_DIR}/df_thesis_films.csv"
SEMANTIC_OUT = f"{PROCESSED_CORE_DIR}/df_semantic_analysis.csv"
REVENUE_OUT = f"{PROCESSED_CORE_DIR}/df_revenue_analysis.csv"
ROI_OUT = f"{PROCESSED_CORE_DIR}/df_roi_analysis.csv"
DISPLAY_OUT = f"{PROCESSED_CORE_DIR}/df_display_candidates.csv"

FUNNEL_OUT = f"{OUTPUT_TABLES_DIR}/filtering_funnel.csv"
DEDUP_OUT = f"{OUTPUT_TABLES_DIR}/dedup_log.csv"
AUDIT_OUT = f"{OUTPUT_TABLES_DIR}/thesis_corpus_audit.csv"
YEAR_COUNTS_OUT = f"{OUTPUT_TABLES_DIR}/movies_per_year_thesis_corpus.csv"
GENRE_COUNTS_OUT = f"{OUTPUT_TABLES_DIR}/primary_genre_counts.csv"

df_thesis_films.to_csv(THESIS_FILMS_OUT, index=False)
df_semantic_analysis.to_csv(SEMANTIC_OUT, index=False)
df_revenue_analysis.to_csv(REVENUE_OUT, index=False)
df_roi_analysis.to_csv(ROI_OUT, index=False)
df_display_candidates.to_csv(DISPLAY_OUT, index=False)

funnel_df.to_csv(FUNNEL_OUT, index=False)
dedup_log_df.to_csv(DEDUP_OUT, index=False)
audit.to_csv(AUDIT_OUT, index=False)
year_counts.to_csv(YEAR_COUNTS_OUT, index=False)
genre_counts.to_csv(GENRE_COUNTS_OUT, index=False)

print("Saved core datasets:")
print("-", THESIS_FILMS_OUT)
print("-", SEMANTIC_OUT)
print("-", REVENUE_OUT)
print("-", ROI_OUT)
print("-", DISPLAY_OUT)

print("\nSaved audit tables:")
print("-", FUNNEL_OUT)
print("-", DEDUP_OUT)
print("-", AUDIT_OUT)
print("-", YEAR_COUNTS_OUT)
print("-", GENRE_COUNTS_OUT)


Saved core datasets:
- /content/drive/MyDrive/thesis_movie_project/processed/core/df_thesis_films.csv
- /content/drive/MyDrive/thesis_movie_project/processed/core/df_semantic_analysis.csv
- /content/drive/MyDrive/thesis_movie_project/processed/core/df_revenue_analysis.csv
- /content/drive/MyDrive/thesis_movie_project/processed/core/df_roi_analysis.csv
- /content/drive/MyDrive/thesis_movie_project/processed/core/df_display_candidates.csv

Saved audit tables:
- /content/drive/MyDrive/thesis_movie_project/outputs/tables/filtering_funnel.csv
- /content/drive/MyDrive/thesis_movie_project/outputs/tables/dedup_log.csv
- /content/drive/MyDrive/thesis_movie_project/outputs/tables/thesis_corpus_audit.csv
- /content/drive/MyDrive/thesis_movie_project/outputs/tables/movies_per_year_thesis_corpus.csv
- /content/drive/MyDrive/thesis_movie_project/outputs/tables/primary_genre_counts.csv


## 11. Quick Note on Tuning
If the network in Notebook 3 ends up being too messy, we don't need to rewrite this code. We can just come back to Section 1, raise the minimum vote counts or word counts, and re-run this notebook to get a stricter dataset.